# Track B - Flood segmentation model (U-Net on Kuro Siwo)
**Before running:** Settings > Accelerator = **GPU P100** (or T4), Internet = **On**, and Add Input > Notebooks > the extraction notebook (the one with `ks_chips`).
Then **Save Version > Save & Run All**. Outputs: `best.pth`, `flood_unet.onnx`, `norm.json`, `metrics.json`, `training_curves.png`.

In [ ]:
!pip install -q segmentation-models-pytorch onnx onnxruntime

In [ ]:
# 1) Find the chips and settings
import os, glob, json, random, time
import numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED, EPOCHS, BATCH, LR, VAL_FRACTION = 42, 15, 16, 5e-4, 0.15
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT = "/kaggle/working"

cands = glob.glob("/kaggle/input/**/ks_chips", recursive=True)
assert cands, "No ks_chips found. Add Input > Notebooks > your extraction notebook."
DATA = max(cands, key=lambda d: len(os.listdir(d)))      # pick the full copy, not a 500-file one
files = sorted(glob.glob(f"{DATA}/*.npz"))
print("Data folder:", DATA, "| chips:", len(files))
assert len(files) > 10000, "Too few chips - you attached the 500-file dataset, attach the notebook output instead"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on GPU in Settings")

In [ ]:
# 2) Split by FLOOD EVENT (not by chip) so validation floods are never seen in training
def event_key(f):
    info = json.loads(str(np.load(f)["info"]))
    for k in ("event", "event_id", "actid", "act_id", "flood_id"):
        if k in info:
            return str(info[k])
    return info.get("sources", {}).get("MS1", {}).get("source_date", "unknown")

print("Top-level info keys example:", list(json.loads(str(np.load(files[0])["info"])).keys()))
t = time.time()
keys = [event_key(f) for f in files]
print(f"Indexed {len(files)} chips in {time.time()-t:.0f}s")

groups = {}
for f, k in zip(files, keys):
    groups.setdefault(k, []).append(f)
events = sorted(groups); random.Random(SEED).shuffle(events)
val_files, val_events = [], []
for e in events:
    if len(val_files) >= VAL_FRACTION * len(files): break
    val_files += groups[e]; val_events.append(e)
val_set = set(val_files)
train_files = [f for f in files if f not in val_set]
print(f"Events: {len(groups)} | train chips {len(train_files)} | val chips {len(val_files)} from {len(val_events)} held-out events")

In [ ]:
# 3) Input recipe - the SAME function is reused later on Google Earth Engine data
NORM = {"channels": ["pre_VV_dB", "pre_VH_dB", "post_VV_dB", "post_VH_dB", "dem_relative", "slope"],
        "sar_clip_db": [-30, 5], "sar_shift": 15, "sar_scale": 10,
        "dem_rel_scale_m": 50, "pixel_size_m": 10, "slope_scale_deg": 45,
        "classes": ["dry", "permanent_water", "flood_water"], "ignore_index": 255}

def make_input(sar_db, dem, norm=NORM):
    """sar_db: (4,H,W) pre VV, pre VH, post VV, post VH in dB. dem: (H,W) metres."""
    sar_db = np.nan_to_num(sar_db, nan=-30.0)
    dem = np.nan_to_num(dem, nan=float(np.nanmean(dem)) if np.isfinite(dem).any() else 0.0)
    lo, hi = norm["sar_clip_db"]
    sar = (np.clip(sar_db, lo, hi) + norm["sar_shift"]) / norm["sar_scale"]
    gy, gx = np.gradient(dem, norm["pixel_size_m"])
    slope = np.degrees(np.arctan(np.hypot(gx, gy))) / norm["slope_scale_deg"]
    dem_rel = np.clip((dem - dem.mean()) / norm["dem_rel_scale_m"], -5, 5)
    return np.concatenate([sar, dem_rel[None], slope[None]]).astype(np.float32)

class Chips(Dataset):
    def __init__(self, files, aug):
        self.files, self.aug = files, aug
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        z = np.load(self.files[i])
        x = make_input(z["sar"].astype(np.float32), z["dem"].astype(np.float32))
        y = z["mask"].astype(np.int64)
        y[(y > 2) & (y != 255)] = 255
        if self.aug:
            k = random.randint(0, 3)
            x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)
            if random.random() < 0.5:
                x, y = x[:, :, ::-1], y[:, ::-1]
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

train_dl = DataLoader(Chips(train_files, True), batch_size=BATCH, shuffle=True, num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)
val_dl = DataLoader(Chips(val_files, False), batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
xb, yb = next(iter(train_dl))
print("Batch:", xb.shape, yb.shape, "| label values:", torch.unique(yb).tolist())

In [ ]:
# 4) Model, loss, metrics
import segmentation_models_pytorch as smp
dev = "cuda"
model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=6, classes=3).to(dev)
ce = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.0, 2.0], device=dev), ignore_index=255)
dice = smp.losses.DiceLoss("multiclass", ignore_index=255)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")

def confusion(pred, y, n=3):
    m = y != 255
    return torch.bincount(y[m] * n + pred[m], minlength=n * n).reshape(n, n).cpu().numpy()

def scores(cm):
    iou = [cm[c, c] / max(cm[c, :].sum() + cm[:, c].sum() - cm[c, c], 1) for c in range(3)]
    tp, fp, fn = cm[2, 2], cm[:, 2].sum() - cm[2, 2], cm[2, :].sum() - cm[2, 2]
    p, r = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"iou_dry": float(iou[0]), "iou_perm_water": float(iou[1]), "iou_flood": float(iou[2]),
            "flood_precision": float(p), "flood_recall": float(r), "flood_f1": float(2 * p * r / max(p + r, 1e-9))}

def baseline_pred(x):
    """Classic rule (our pipeline's thresholding): dark after AND got darker -> flood."""
    pre_vv = x[:, 0] * NORM["sar_scale"] - NORM["sar_shift"]
    post_vv = x[:, 2] * NORM["sar_scale"] - NORM["sar_shift"]
    pred = torch.zeros_like(pre_vv, dtype=torch.long)
    pred[(pre_vv < -18) & (post_vv < -18)] = 1
    pred[(post_vv < -18) & (post_vv - pre_vv < -3)] = 2
    return pred

In [ ]:
# 5) Train
history, best_iou = [], -1
for ep in range(1, EPOCHS + 1):
    model.train(); t0, run = time.time(), 0.0
    for i, (x, y) in enumerate(train_dl):
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            out = model(x)
            loss = ce(out, y) + dice(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
    model.eval(); cm = np.zeros((3, 3), np.int64)
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for x, y in val_dl:
            cm += confusion(model(x.to(dev)).argmax(1), y.to(dev))
    s = scores(cm); s.update(epoch=ep, train_loss=run / len(train_dl), minutes=(time.time() - t0) / 60)
    history.append(s)
    flag = ""
    if s["iou_flood"] > best_iou:
        best_iou = s["iou_flood"]; torch.save(model.state_dict(), f"{OUT}/best.pth"); flag = "  <- best, saved"
    print(f"ep {ep:2d} | loss {s['train_loss']:.3f} | flood IoU {s['iou_flood']:.3f} F1 {s['flood_f1']:.3f} "
          f"P {s['flood_precision']:.3f} R {s['flood_recall']:.3f} | {s['minutes']:.1f} min{flag}", flush=True)

In [ ]:
# 6) Final evaluation: U-Net vs simple thresholding baseline on held-out events
model.load_state_dict(torch.load(f"{OUT}/best.pth")); model.eval()
cm_m, cm_b = np.zeros((3, 3), np.int64), np.zeros((3, 3), np.int64)
with torch.no_grad(), torch.amp.autocast("cuda"):
    for x, y in val_dl:
        x, y = x.to(dev), y.to(dev)
        cm_m += confusion(model(x).argmax(1), y)
        cm_b += confusion(baseline_pred(x.float()), y)
unet, base = scores(cm_m), scores(cm_b)
print(f"{'metric':<18}{'baseline':>10}{'U-Net':>10}")
for k in unet:
    print(f"{k:<18}{base[k]:>10.3f}{unet[k]:>10.3f}")
json.dump({"unet_val": unet, "baseline_val": base, "history": history, "val_events": val_events,
           "n_train": len(train_files), "n_val": len(val_files)}, open(f"{OUT}/metrics.json", "w"), indent=2)
json.dump(NORM, open(f"{OUT}/norm.json", "w"), indent=2)

In [ ]:
# 7) Training curves
import matplotlib.pyplot as plt
ep = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [h["train_loss"] for h in history]); ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch")
for k in ("iou_flood", "flood_f1", "flood_precision", "flood_recall"):
    ax[1].plot(ep, [h[k] for h in history], label=k)
ax[1].axhline(base["flood_f1"], ls="--", c="gray", label="baseline F1")
ax[1].set_title("Validation (held-out events)"); ax[1].legend(); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(f"{OUT}/training_curves.png", dpi=150); plt.show()

In [ ]:
# 8) Export to ONNX so the live demo runs on CPU (no GPU needed)
import onnxruntime as ort
m = model.float().cpu().eval()
dummy = torch.randn(1, 6, 256, 256)
torch.onnx.export(m, dummy, f"{OUT}/flood_unet.onnx", input_names=["x"], output_names=["logits"],
                  dynamic_axes={"x": {0: "b", 2: "h", 3: "w"}, "logits": {0: "b", 2: "h", 3: "w"}}, opset_version=17)
sess = ort.InferenceSession(f"{OUT}/flood_unet.onnx")
diff = np.abs(sess.run(None, {"x": dummy.numpy()})[0] - m(dummy).detach().numpy()).max()
print("ONNX export OK, max difference vs PyTorch:", float(diff))
print(sorted(os.listdir(OUT)))